# Round 1 ML Strategy
This notebook trains and evaluates lightweight machine-learning signals on the official `data/ROUND_1` order books, then compares the final deployed trader in `ROUND_1/ml_round1_trader.py` against the strongest pre-ML baselines already present in the repo.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'ROUND_1'
RUNS = ROOT / 'runs'
DAYS = [-2, -1, 0]
PRODUCTS = ['ASH_COATED_OSMIUM', 'INTARIAN_PEPPER_ROOT']
ROOT

PosixPath('/home/tk/imc-prosperity-4')

In [2]:
def load_product(day: int, product: str) -> pd.DataFrame:
    df = pd.read_csv(DATA / f'prices_round_1_day_{day}.csv', sep=';')
    sub = df[df['product'] == product].copy().sort_values('timestamp').reset_index(drop=True)
    sub = sub[sub['bid_price_1'].notna() & sub['ask_price_1'].notna()].reset_index(drop=True)
    return sub

def build_full_features(sub: pd.DataFrame) -> pd.DataFrame:
    df = sub.copy()
    df['spread'] = df['ask_price_1'] - df['bid_price_1']
    df['micro'] = (
        df['ask_price_1'] * df['bid_volume_1'].fillna(0)
        + df['bid_price_1'] * df['ask_volume_1'].fillna(0)
    ) / (df['bid_volume_1'].fillna(0) + df['ask_volume_1'].fillna(0)).replace(0, np.nan)
    for level in [1, 2, 3]:
        denom = (df[f'bid_volume_{level}'].fillna(0) + df[f'ask_volume_{level}'].fillna(0)).replace(0, np.nan)
        df[f'imbalance_{level}'] = (df[f'bid_volume_{level}'].fillna(0) - df[f'ask_volume_{level}'].fillna(0)) / denom
        df[f'bid_gap_{level}'] = df['mid_price'] - df[f'bid_price_{level}']
        df[f'ask_gap_{level}'] = df[f'ask_price_{level}'] - df['mid_price']
    df['micro_gap'] = df['micro'] - df['mid_price']
    df['time_idx'] = np.arange(len(df))
    df['time_remaining'] = len(df) - 1 - df['time_idx']
    for lag in [1, 2, 5, 10, 20, 50, 100]:
        df[f'mid_ret_lag_{lag}'] = df['mid_price'] - df['mid_price'].shift(lag)
        df[f'spread_lag_{lag}'] = df['spread'].shift(lag)
        df[f'micro_gap_lag_{lag}'] = df['micro_gap'].shift(lag)
        df[f'imbalance_1_lag_{lag}'] = df['imbalance_1'].shift(lag)
    return df

def build_osmium_linear_features(sub: pd.DataFrame) -> pd.DataFrame:
    df = sub.copy()
    df['spread'] = (df['ask_price_1'] - df['bid_price_1']) / 10.0
    df['micro'] = (
        df['ask_price_1'] * df['bid_volume_1'].fillna(0)
        + df['bid_price_1'] * df['ask_volume_1'].fillna(0)
    ) / (df['bid_volume_1'].fillna(0) + df['ask_volume_1'].fillna(0)).replace(0, np.nan)
    denom = (df['bid_volume_1'].fillna(0) + df['ask_volume_1'].fillna(0)).replace(0, np.nan)
    df['imbalance_1'] = (df['bid_volume_1'].fillna(0) - df['ask_volume_1'].fillna(0)) / denom
    df['micro_gap'] = (df['micro'] - df['mid_price']) / 5.0
    for lag in [1, 2, 5]:
        df[f'mid_ret_lag_{lag}'] = (df['mid_price'] - df['mid_price'].shift(lag)) / 5.0
        df[f'imbalance_1_lag_{lag}'] = df['imbalance_1'].shift(lag)
        df[f'micro_gap_lag_{lag}'] = df['micro_gap'].shift(lag)
    return df


## Strongest Existing Baseline
Before adding any ML, the repo already contained a spread-based Pepper+Osmium family. We can recover the best complete 3-day non-ML trader directly from `runs/*/metrics.json`.

In [3]:
rows = []
for metrics_path in RUNS.glob('*/metrics.json'):
    metrics = json.loads(metrics_path.read_text())
    trader_path = metrics.get('trader_path', '')
    run_id = metrics.get('run_id', '')
    if run_id.startswith('ml') or '/.ml_tmp' in trader_path or 'ml_round1_trader.py' in trader_path:
        continue
    rows.append({
        'run_id': run_id,
        'day': metrics['day'],
        'trader_path': trader_path,
        'final_pnl_total': metrics['final_pnl_total'],
    })
runs_df = pd.DataFrame(rows)
baseline = (runs_df.groupby('trader_path')
    .agg(days=('day', 'nunique'), avg_pnl=('final_pnl_total', 'mean'), min_pnl=('final_pnl_total', 'min'))
    .query('days == 3')
    .sort_values(['avg_pnl', 'min_pnl'], ascending=False)
    .head(10))
baseline

,days,avg_pnl,min_pnl
trader_path,,,
ROUND_1/.pepper_combo2/cand_100.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_102.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_104.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_106.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_108.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_110.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_112.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_114.py,3,94198.666667,93548.0
ROUND_1/.pepper_combo2/cand_116.py,3,94198.666667,93548.0


## Model Benchmark
We compare a few real ML models out-of-sample with leave-one-day-out validation. For Osmium, the useful target is short-horizon mid-price change; for Pepper, the useful target is longer-horizon drift because the dominant edge is carrying inventory into the close.

In [4]:
def benchmark_product(product: str, horizons):
    feature_frames = {}
    for day in DAYS:
        feature_frames[day] = build_full_features(load_product(day, product))
    models = {
        'ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10.0)),
        'rf': make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=120, max_depth=6, random_state=0, n_jobs=-1)),
        'hgb': make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=200, random_state=0)),
    }
    benchmark_rows = []
    for horizon in horizons:
        prepared = []
        for day in DAYS:
            frame = feature_frames[day].copy()
            frame['target'] = frame['mid_price'].shift(-horizon) - frame['mid_price']
            frame = frame.dropna(subset=['target']).reset_index(drop=True)
            prepared.append((day, frame))
        feature_cols = [c for c in prepared[0][1].columns if c not in {'day', 'timestamp', 'product', 'profit_and_loss', 'target'}]
        for model_name, model in models.items():
            preds, ys = [], []
            for test_day in DAYS:
                train = pd.concat([frame for day, frame in prepared if day != test_day], ignore_index=True)
                test = next(frame for day, frame in prepared if day == test_day)
                model.fit(train[feature_cols], train['target'])
                preds.append(model.predict(test[feature_cols]))
                ys.append(test['target'].to_numpy())
            y = np.concatenate(ys)
            p = np.concatenate(preds)
            benchmark_rows.append({
                'product': product,
                'horizon': horizon,
                'model': model_name,
                'r2': r2_score(y, p),
                'corr': np.corrcoef(y, p)[0, 1],
                'pred_abs_mean': np.mean(np.abs(p)),
            })
    return pd.DataFrame(benchmark_rows)

benchmark_df = pd.concat([
    benchmark_product('ASH_COATED_OSMIUM', [1, 5, 10]),
    benchmark_product('INTARIAN_PEPPER_ROOT', [10, 50]),
], ignore_index=True)
benchmark_df.sort_values(['product', 'horizon', 'r2'], ascending=[True, True, False]).reset_index(drop=True)

,product,horizon,model,r2,corr,pred_abs_mean
0,ASH_COATED_OSMIUM,1,rf,0.476049,0.689977,0.765090
1,ASH_COATED_OSMIUM,1,hgb,0.475743,0.689956,0.737118
2,ASH_COATED_OSMIUM,1,ridge,0.465883,0.682561,0.797773
3,ASH_COATED_OSMIUM,5,rf,0.419062,0.647350,0.749560
4,ASH_COATED_OSMIUM,5,hgb,0.417929,0.646516,0.742071
5,ASH_COATED_OSMIUM,5,ridge,0.412007,0.641889,0.788752
6,ASH_COATED_OSMIUM,10,rf,0.380366,0.616761,0.779795
7,ASH_COATED_OSMIUM,10,hgb,0.376744,0.613845,0.802956
8,ASH_COATED_OSMIUM,10,ridge,0.375766,0.613008,0.817372
9,INTARIAN_PEPPER_ROOT,10,hgb,0.482821,0.695093,1.288942


## Final Submission-Safe Osmium Model
The deployed trader uses a small linear model so the runtime stays within the Prosperity library constraints. The coefficients below are fitted offline, then embedded directly into `ROUND_1/ml_round1_trader.py`.

In [5]:
osmium_rows = []
for day in DAYS:
    frame = build_osmium_linear_features(load_product(day, 'ASH_COATED_OSMIUM'))
    frame['target'] = frame['mid_price'].shift(-5) - frame['mid_price']
    frame['day'] = day
    osmium_rows.append(frame)
osmium_df = pd.concat(osmium_rows, ignore_index=True)
feature_cols = [
    'spread', 'imbalance_1', 'micro_gap', 'mid_ret_lag_1', 'mid_ret_lag_2', 'mid_ret_lag_5',
    'imbalance_1_lag_1', 'imbalance_1_lag_2', 'imbalance_1_lag_5',
    'micro_gap_lag_1', 'micro_gap_lag_2', 'micro_gap_lag_5',
]
osmium_df = osmium_df.dropna(subset=feature_cols + ['target']).reset_index(drop=True)
ridge = Ridge(alpha=1.0)
preds, ys = [], []
for test_day in DAYS:
    train = osmium_df[osmium_df['day'] != test_day]
    test = osmium_df[osmium_df['day'] == test_day]
    ridge.fit(train[feature_cols], train['target'])
    preds.append(ridge.predict(test[feature_cols]))
    ys.append(test['target'].to_numpy())
y = np.concatenate(ys)
p = np.concatenate(preds)
print({'osmium_r2': r2_score(y, p), 'osmium_corr': np.corrcoef(y, p)[0, 1]})
ridge.fit(osmium_df[feature_cols], osmium_df['target'])
pd.Series(ridge.coef_, index=feature_cols).sort_values(key=lambda s: s.abs(), ascending=False).to_frame('coefficient')

{'osmium_r2': 0.3902228686982053, 'osmium_corr': np.float64(0.6246783226704238)}


,coefficient
imbalance_1,4.599323
imbalance_1_lag_1,2.866034
imbalance_1_lag_2,2.241078
micro_gap,-1.742129
mid_ret_lag_1,-1.322975
imbalance_1_lag_5,1.190819
micro_gap_lag_1,-1.091608
mid_ret_lag_2,-1.007575
micro_gap_lag_2,-0.898534
micro_gap_lag_5,-0.517795


## Final Backtest
These are the checked-in results for `ROUND_1/ml_round1_trader.py`, using the Rust backtester on all three official Round 1 days.

In [6]:
final_rows = []
for day in DAYS:
    run_id = f'ml_final_d{day}'
    metrics = json.loads((RUNS / run_id / 'metrics.json').read_text())
    final_rows.append({
        'day': day,
        'total_pnl': metrics['final_pnl_total'],
        'osmium_pnl': metrics['final_pnl_by_product']['ASH_COATED_OSMIUM'],
        'pepper_pnl': metrics['final_pnl_by_product']['INTARIAN_PEPPER_ROOT'],
        'own_trade_count': metrics['own_trade_count'],
    })
final_df = pd.DataFrame(final_rows).sort_values('day').reset_index(drop=True)
final_df.loc['avg'] = ['avg', final_df['total_pnl'].mean(), final_df['osmium_pnl'].mean(), final_df['pepper_pnl'].mean(), final_df['own_trade_count'].mean()]
final_df

,day,total_pnl,osmium_pnl,pepper_pnl,own_trade_count
0,-2,95794.0,16233.0,79561.0,509.000000
1,-1,98370.0,19136.0,79234.0,550.000000
2,0,98378.0,18928.0,79450.0,566.000000
avg,avg,97514.0,18099.0,79415.0,541.666667


In [7]:
best_baseline = baseline.reset_index().iloc[0]
comparison = pd.DataFrame([
    {
        'strategy': 'best pre-ML baseline',
        'trader_path': best_baseline['trader_path'],
        'avg_pnl': best_baseline['avg_pnl'],
        'min_pnl': best_baseline['min_pnl'],
    },
    {
        'strategy': 'ml_round1_trader',
        'trader_path': 'ROUND_1/ml_round1_trader.py',
        'avg_pnl': final_df.loc[final_df['day'] == 'avg', 'total_pnl'].iloc[0],
        'min_pnl': final_df.loc[final_df['day'] != 'avg', 'total_pnl'].min(),
    },
])
comparison['avg_pnl_lift'] = comparison['avg_pnl'] - comparison['avg_pnl'].iloc[0]
comparison

,strategy,trader_path,avg_pnl,min_pnl,avg_pnl_lift
0,best pre-ML baseline,ROUND_1/.pepper_combo2/cand_100.py,94198.666667,93548.0,0.000000
1,ml_round1_trader,ROUND_1/ml_round1_trader.py,97514.000000,95794.0,3315.333333


## Result
The final strategy keeps Pepper simple and inventory-heavy because the learned end-value slope is consistently positive, then uses a lightweight ML fair-value model to shift Osmium quoting and inventory unwind. That combination materially improves Round 1 PnL over the earlier heuristic baselines while staying submission-safe.